Replication of the 2017 paper: Attention Is All You Need per the tutorial here: 

https://nlp.seas.harvard.edu/2018/04/03/attention.html


### Library versions from original
pytorch 0.4.1 <br>
numpy=1.15.4 matplotlib=2.2.3 seaborn=0.8.1 cython=0.29 <br>
spacy==2.1.8 <br>

In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, copy, time
from torch.autograd import Variable
import matplotlib.pyplot as plt
import seaborn
seaborn.set_context(context="talk")
%matplotlib inline

# Model Build Starts Here

In [6]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many 
    other models.
    """
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
    
        
    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask,
                            tgt, tgt_mask)
    

In [7]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)

In [8]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [9]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [10]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
        
    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [11]:
class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))

In [12]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

In [13]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
        
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

In [14]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
 
    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

In [15]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
    return torch.from_numpy(subsequent_mask) == 0

In [16]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) \
             / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = F.softmax(scores, dim = -1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [20]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        # We assume d_v always equals d_k
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
        
    def forward(self, query, key, value, mask=None):
        "Implements Figure 2"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        
        # 1) Do all the linear projections in batch from d_model => h x d_k 
        query, key, value = \
            [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
             for l, x in zip(self.linears, (query, key, value))]
        
        # 2) Apply attention on all the projected vectors in batch. 
        x, self.attn = attention(query, key, value, mask=mask, 
                                 dropout=self.dropout)
        
        # 3) "Concat" using a view and apply a final linear. 
        x = x.transpose(1, 2).contiguous() \
             .view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)

In [18]:
class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))

In [19]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

In [21]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) *
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + Variable(self.pe[:, :x.size(1)], 
                         requires_grad=False)
        return self.dropout(x)
        

In [22]:
def make_model(src_vocab, tgt_vocab, N=6, 
               d_model=512, d_ff=2048, h=8, dropout=0.1):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), 
                             c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab))
    
    # This was important from their code. 
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform(p)
    return model

In [23]:
def make_model(src_vocab, tgt_vocab, N=6, 
               d_model=512, d_ff=2048, h=8, dropout=0.1):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), 
                             c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab))
    
    # This was important from their code. 
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform(p)
    return model

In [24]:
tmp_model = make_model(10, 10, 2)

/tmp/ipykernel_3171120/2289673833.py:20: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
  nn.init.xavier_uniform(p)


# Training Setup Starts Here

In [ ]:
import os
import time
import random
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

# === DDP CHANGE ===
def setup_distributed():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank

# === DDP CHANGE ===
def cleanup_distributed():
    dist.destroy_process_group()

# === DDP CHANGE ===
def is_main():
    return dist.get_rank() == 0

In [ ]:
class Batch:
    "Object for holding a batch of data with mask during training."
    def __init__(self, src, trg=None, pad=0):
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)
        if trg is not None:
            self.trg = trg[:, :-1]
            self.trg_y = trg[:, 1:]
            self.trg_mask = \
                self.make_std_mask(self.trg, pad)
            self.ntokens = (self.trg_y != pad).sum()
    
    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & Variable(
            subsequent_mask(tgt.size(-1)).type_as(tgt_mask))
        return tgt_mask

In [ ]:
def run_epoch(data_iter, model, loss_compute):
    "Standard Training and Logging Function"
    start = time.time()
    total_tokens = 0
    total_loss = 0
    tokens = 0
    for i, batch in enumerate(data_iter):
        
        out = model.forward(batch.src, batch.trg, 
                            batch.src_mask, batch.trg_mask)
        loss = loss_compute(out, batch.trg_y, batch.ntokens)
        total_loss += loss
        total_tokens += batch.ntokens
        tokens += batch.ntokens
        if i % 50 == 1:

            elapsed = time.time() - start
            print("Epoch Step: %d Loss: %f Tokens per Sec: %f" % (i, (loss.float() / batch.ntokens.float()).item(),tokens.float().item() / elapsed))
            start = time.time()
            tokens = 0
    return total_loss.float() / total_tokens.float()

In [ ]:
global max_src_in_batch, max_tgt_in_batch
def batch_size_fn(new, count, sofar):
    "Keep augmenting batch and calculate total number of tokens + padding."
    global max_src_in_batch, max_tgt_in_batch
    if count == 1:
        max_src_in_batch = 0
        max_tgt_in_batch = 0
    max_src_in_batch = max(max_src_in_batch,  len(new.src))
    max_tgt_in_batch = max(max_tgt_in_batch,  len(new.trg) + 2)
    src_elements = count * max_src_in_batch
    tgt_elements = count * max_tgt_in_batch
    return max(src_elements, tgt_elements)

In [ ]:
class NoamOpt:
    "Optim wrapper that implements rate."
    def __init__(self, model_size, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0
        
    def step(self):
        "Update parameters and rate"
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()
        
    def rate(self, step = None):
        "Implement `lrate` above"
        if step is None:
            step = self._step
        return self.factor * \
            (self.model_size ** (-0.5) *
            min(step ** (-0.5), step * self.warmup ** (-1.5)))
        
def get_std_opt(model):
    return NoamOpt(model.src_embed[0].d_model, 2, 4000,
            torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))


In [ ]:
class LabelSmoothing(nn.Module):
    "Implement label smoothing."
    def __init__(self, size, padding_idx, smoothing=0.0):
        super(LabelSmoothing, self).__init__()
        self.criterion = nn.KLDivLoss(size_average=False)
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None
        
    def forward(self, x, target):
        assert x.size(1) == self.size
        true_dist = x.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = target == self.padding_idx
        true_dist[mask] = 0.0  # directly zero out padding positions

        self.true_dist = true_dist
        return self.criterion(x, Variable(true_dist, requires_grad=False))

In [ ]:
def data_gen(V, batch, nbatches):
    "Generate random data for a src-tgt copy task."
    for i in range(nbatches):
        data = torch.randint(1, V, (batch, 10))  # random token indices
        src = data.long()  # cast here
        tgt = data.long()  # cast here
        yield Batch(src, tgt, 0)

In [ ]:
class SimpleLossCompute:
    "A simple loss compute and train function."
    def __init__(self, generator, criterion, opt=None):
        self.generator = generator
        self.criterion = criterion
        self.opt = opt
        
    def __call__(self, x, y, norm):
        norm = norm.float()
        x = self.generator(x)

        loss = self.criterion(x.contiguous().view(-1, x.size(-1)),
                              y.contiguous().view(-1)) / norm
        
        if self.opt is not None:
            loss.backward()
            self.opt.step()
            self.opt.optimizer.zero_grad()

        return loss.item() * norm

In [37]:
V = 11
criterion = LabelSmoothing(size=V, padding_idx=0, smoothing=0.0)
model = make_model(V, V, N=2)


model_opt = NoamOpt(model.src_embed[0].d_model, 1, 400,
        torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

for epoch in range(10):
    model.train()
    run_epoch(data_gen(V, 30, 20), model, 
              SimpleLossCompute(model.generator, criterion, model_opt))
    model.eval()
    print(run_epoch(data_gen(V, 30, 5), model, 
                    SimpleLossCompute(model.generator, criterion, None)))

/tmp/ipykernel_3171120/2289673833.py:20: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
  nn.init.xavier_uniform(p)


Epoch Step: 1 Loss: 2.926784 Tokens per Sec: 1872.844222
Epoch Step: 1 Loss: 1.868255 Tokens per Sec: 6608.845821
tensor(1.9122)
Epoch Step: 1 Loss: 2.018247 Tokens per Sec: 3895.899725
Epoch Step: 1 Loss: 1.749661 Tokens per Sec: 8470.711152
tensor(1.7105)
Epoch Step: 1 Loss: 1.938289 Tokens per Sec: 2978.106124
Epoch Step: 1 Loss: 1.651771 Tokens per Sec: 8219.976700
tensor(1.6755)
Epoch Step: 1 Loss: 1.857871 Tokens per Sec: 1834.680560
Epoch Step: 1 Loss: 1.460035 Tokens per Sec: 155.353106
tensor(1.3978)
Epoch Step: 1 Loss: 1.705392 Tokens per Sec: 1057.386082
Epoch Step: 1 Loss: 1.146457 Tokens per Sec: 5873.948774
tensor(1.0970)
Epoch Step: 1 Loss: 1.280600 Tokens per Sec: 1666.715107
Epoch Step: 1 Loss: 0.863452 Tokens per Sec: 424.456356
tensor(0.7807)
Epoch Step: 1 Loss: 1.084462 Tokens per Sec: 883.414674
Epoch Step: 1 Loss: 0.472169 Tokens per Sec: 8613.876831
tensor(0.5103)
Epoch Step: 1 Loss: 0.531562 Tokens per Sec: 2405.463357
Epoch Step: 1 Loss: 0.293064 Tokens per Sec

In [ ]:
def greedy_decode(model, src, src_mask, max_len, start_symbol, eos_symbol=None):
    memory = model.encode(src, src_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)
    for i in range(max_len-1):
        out = model.decode(memory, src_mask, 
                           Variable(ys), 
                           Variable(subsequent_mask(ys.size(1))
                                    .type_as(src.data)))
        prob = model.generator(out[:, -1])

        _, next_word = torch.max(prob, dim = 1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, 
                        torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=1)
        if eos_symbol is not None and next_word == eos_symbol:
            break
    return ys
model.eval()
src = Variable(torch.LongTensor([[1,2,3,5,5,6,7,8,10,10]]) )
src_mask = Variable(torch.ones(1, 1, 10) )
print(greedy_decode(model, src, src_mask, max_len=10, start_symbol=1))

tensor([[-14.8333,  -5.0058,  -0.2077,  -2.1063,  -4.9928,  -3.8258,  -4.4652,
          -6.0660,  -6.2668,  -5.5480,  -4.5126]],
       grad_fn=<LogSoftmaxBackward0>)
tensor([[-17.1678, -10.1703,  -7.6848,  -0.0585,  -7.3388,  -2.9377,  -6.3454,
          -9.4283,  -8.9847,  -7.7854,  -8.1216]],
       grad_fn=<LogSoftmaxBackward0>)
tensor([[-2.0756e+01, -1.1829e+01, -1.0524e+01, -7.9392e+00, -9.6665e+00,
         -8.8545e-04, -8.2302e+00, -1.0537e+01, -1.1280e+01, -9.5779e+00,
         -9.7893e+00]], grad_fn=<LogSoftmaxBackward0>)
tensor([[-1.8046e+01, -9.6780e+00, -7.4606e+00, -4.7844e+00, -7.1045e+00,
         -1.4577e-02, -5.7833e+00, -8.5853e+00, -8.8330e+00, -7.0820e+00,
         -7.8224e+00]], grad_fn=<LogSoftmaxBackward0>)
tensor([[-15.8969,  -8.5312,  -7.7232,  -5.3622,  -4.5253,  -2.2928,  -0.1462,
          -6.4704,  -6.1710,  -4.2725,  -6.6320]],
       grad_fn=<LogSoftmaxBackward0>)
tensor([[-16.9418,  -9.0046,  -7.9491,  -7.7032,  -6.7401, -10.8565,  -8.3966,
          -

In [ ]:
def beam_search_decode(model, src, src_mask, max_len, start_symbol,
                      beam_size=4, alpha=0.6, eos_symbol=None):

    memory = model.encode(src, src_mask)

    beams = [(torch.ones(1,1).fill_(start_symbol).type_as(src), 0.0, False)]

    def length_penalty(length):
        return ((5 + length) ** alpha) / ((5 + 1) ** alpha)

    for _ in range(max_len - 1):

        new_beams = []

        all_finished = True

        for seq, score, finished in beams:

            if finished:
                new_beams.append((seq, score, True))
                continue

            all_finished = False

            out = model.decode(
                memory,
                src_mask,
                Variable(seq),
                Variable(subsequent_mask(seq.size(1)).type_as(src))
            )

            prob = model.generator(out[:, -1])
            topk_log_probs, topk_ids = torch.topk(prob, beam_size)
            # probs = torch.exp(prob)
            # print(probs[0, eos_symbol])

            for k in range(beam_size):
                next_word = topk_ids[0][k].item()
                next_score = score + topk_log_probs[0][k].item()

                new_seq = torch.cat([
                    seq,
                    torch.ones(1,1).type_as(src).fill_(next_word)
                ], dim=1)

                # mark as finished if EOS generated
                is_finished = (eos_symbol is not None and next_word == eos_symbol)

                new_beams.append((new_seq, next_score, is_finished))

        # --- early stopping if all beams finished ---
        if all_finished:
            break

        # --- rank beams with length penalty ---
        new_beams = sorted(
            new_beams,
            key=lambda x: x[1] / length_penalty(x[0].size(1)),
            reverse=True
        )

        beams = new_beams[:beam_size]

    # return best sequence
    return beams[0][0]



# model.eval()
# src = Variable(torch.LongTensor([[1,2,3,5,5,6,7,8,10,10]]) )
# src_mask = Variable(torch.ones(1, 1, 10) )

# print(beam_search_decode(model, src, src_mask, max_len=10, start_symbol=1))

In [ ]:
import json
import re
from collections import defaultdict, Counter


class BPETokenizer:

    def __init__(self, vocab_size=37000):
        self.vocab_size = vocab_size
        self.merges = []
        self.vocab = []
        self.word_freqs = None
        self.pattern = re.compile(r"\w+|[^\w\s]")

    def pre_tokenize(self, text):
        return self.pattern.findall(text)


    def build_word_freqs(self, corpus):

        word_freqs = Counter()

        for sentence in corpus:
            for w in self.pre_tokenize(sentence):
                word_freqs[w] += 1

        self.word_freqs = word_freqs


    def initialize_splits(self):

        splits = {}
        pair_freqs = defaultdict(int)
        pair_to_words = defaultdict(set)

        for word, freq in self.word_freqs.items():

            split = list(word) + ["</w>"]
            splits[word] = split

            for i in range(len(split) - 1):

                pair = (split[i], split[i+1])
                pair_freqs[pair] += freq
                pair_to_words[pair].add(word)

        return splits, pair_freqs, pair_to_words



    def merge_pair(self, pair, splits, pair_freqs, pair_to_words):

        a, b = pair
        new_symbol = a + b

        affected_words = pair_to_words[pair]

        for word in list(affected_words):

            split = splits[word]
            freq = self.word_freqs[word]

            i = 0
            new_split = []

            while i < len(split):

                if i < len(split)-1 and split[i] == a and split[i+1] == b:

                    if i > 0:
                        prev_pair = (split[i-1], split[i])
                        pair_freqs[prev_pair] -= freq
                        pair_to_words[prev_pair].discard(word)

                    if i < len(split)-2:
                        next_pair = (split[i+1], split[i+2])
                        pair_freqs[next_pair] -= freq
                        pair_to_words[next_pair].discard(word)

                    new_split.append(new_symbol)
                    i += 2

                else:
                    new_split.append(split[i])
                    i += 1

            splits[word] = new_split

            for j in range(len(new_split)-1):
                p = (new_split[j], new_split[j+1])
                pair_freqs[p] += freq
                pair_to_words[p].add(word)

        pair_freqs[pair] = 0
        pair_to_words[pair].clear()


    def train(self, corpus):

        print("Building word frequencies...")
        self.build_word_freqs(corpus)

        splits, pair_freqs, pair_to_words = self.initialize_splits()

        alphabet = set()
        for word in self.word_freqs:
            alphabet.update(word)

        self.vocab = ["<blank>", "<s>", "</s>"] + sorted(alphabet) + ["</w>"]

        print("Training BPE...")

        while len(self.vocab) < self.vocab_size:
            if (len(self.vocab)%1000==0):
                print(f"{len(self.vocab)}/{self.vocab_size}")
            best = max(pair_freqs, key=pair_freqs.get)

            if pair_freqs[best] == 0:
                break

            self.merges.append(best)
            self.vocab.append(best[0] + best[1])

            self.merge_pair(best, splits, pair_freqs, pair_to_words)

        self.merge_ranks = {pair: i for i, pair in enumerate(self.merges)}

        print("BPE training finished")
        print("Vocab size:", len(self.vocab))



    def tokenize_word(self, word):

        word = list(word) + ["</w>"]

        pairs = {(word[i], word[i+1]) for i in range(len(word)-1)}

        while True:

            candidate = None
            best_rank = float("inf")

            for p in pairs:
                if p in self.merge_ranks and self.merge_ranks[p] < best_rank:
                    best_rank = self.merge_ranks[p]
                    candidate = p

            if candidate is None:
                break

            a, b = candidate
            new_token = a + b

            new_word = []
            i = 0

            while i < len(word):

                if i < len(word)-1 and word[i] == a and word[i+1] == b:
                    new_word.append(new_token)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1

            word = new_word
            pairs = {(word[i], word[i+1]) for i in range(len(word)-1)}

        return word


    def tokenize(self, text):

        words = self.pre_tokenize(text)
        tokens = []

        for word in words:

            split = self.tokenize_word(word)

            for i, token in enumerate(split):

                tokens.append(token)

        return tokens


    def detokenize(self, tokens):

        sentence = "".join(tokens)
        sentence = sentence.replace("</w>", " ")\
            .replace("<s>", "").replace("</s>", "")\
            .replace("<blank>", "")
        return sentence


    def save(self, path):

        with open(path, "w") as f:
            json.dump({
                "vocab": self.vocab,
                "merges": self.merges
            }, f)


    def load(self, path):

        with open(path) as f:
            tok = json.load(f)

        self.vocab = tok["vocab"]
        self.merges = [tuple(m) for m in tok["merges"]]
        self.merge_ranks = {pair: i for i, pair in enumerate(self.merges)}


In [ ]:
from datasets import load_dataset #load the dataset from HuggingFace datasets library

# Load WMT14 German-English
ds = load_dataset("wmt14", "de-en")

splits = ["train", "validation", "test"]

for split in splits:
    de_path = f"wmt14.{split}.de"
    en_path = f"wmt14.{split}.en"

    with open(de_path, "w", encoding="utf-8") as f_de, \
         open(en_path, "w", encoding="utf-8") as f_en:
        for ex in ds[split]:
            f_de.write(ex["translation"]["de"] + "\n")
            f_en.write(ex["translation"]["en"] + "\n")

    print(f"Saved {split} split: {de_path}, {en_path}")


In [ ]:
BOS_WORD = '<s>'
EOS_WORD = '</s>'
BLANK_WORD = "<blank>" 
 
if False: #Set to true to train tokenizer from scratch on WMT14 data (takes a while)
    bpe_tokenizer = BPETokenizer(vocab_size=37000) 

    def load_sentences(filepaths): 
        sentences = [] 
        for path in filepaths: 
            with open(path, encoding="utf-8") as f: 
                sentences.extend(line.strip() for line in f if line.strip()) 
        return sentences 

    train_sentences = load_sentences([ "wmt14/wmt14.train.de", "wmt14/wmt14.train.en" ]) 
    bpe_tokenizer.train(train_sentences) 
    bpe_tokenizer.save("wmt_ruleset.json")

In [44]:
bpe_tokenizer = BPETokenizer()

bpe_tokenizer.load("wmt_ruleset.json")

bpe_tokenizer.tokenize("I am snug like a bug in a rug")

['I', 'am', 's@@', 'nu@@', 'g', 'like', 'a', 'bug', 'in', 'a', 'ru@@', 'g']

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchtext import data

SRC = data.Field(
    tokenize=lambda text: bpe_tokenizer.tokenize(text),
    pad_token=BLANK_WORD
)

TGT = data.Field(
    tokenize=lambda text: bpe_tokenizer.tokenize(text),
    init_token=BOS_WORD,
    eos_token=EOS_WORD,
    pad_token=BLANK_WORD
)

MAX_LEN = 120

def filter_pred(example):
    return len(example.src) <= MAX_LEN and len(example.trg) <= MAX_LEN


# =========================================================
# === DATA LOADING ========================================
# =========================================================

def load_parallel_corpus(de_path, en_path, shuffle=False, pool_size=1000, seed=42):
    examples = []
    with open(de_path, encoding="utf-8") as f_de, open(en_path, encoding="utf-8") as f_en:
        for src_line, trg_line in zip(f_de, f_en):
            src_line = src_line.strip()
            trg_line = trg_line.strip()
            if src_line and trg_line:
                examples.append(
                    data.Example.fromlist(
                        [src_line, trg_line],
                        fields=[('src', SRC), ('trg', TGT)]
                    )
                )

    if shuffle:
        # --- 1. Global sort by length (stratification) ---
        examples.sort(key=lambda x: max(len(x.src), len(x.trg)))
        # --- 2. Partition into pools ---
        pools = [ examples[i:i + pool_size] for i in range(0, len(examples), pool_size) ]
        rng = random.Random(seed)
        # --- 3. Shuffle within each pool ---
        for pool in pools:
            rng.shuffle(pool)
        # --- 4. Shuffle pool order (prevents curriculum bias) ---
        rng.shuffle(pools)
        # --- 5. Flatten back to dataset ---
        examples = [ex for pool in pools for ex in pool]

    return examples

BPE vocab size: 37000


In [ ]:
class TranslationVocab:
    def __init__(self, vocab):
        self.itos = list(dict.fromkeys(vocab))
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

SRC.vocab = TranslationVocab(bpe_tokenizer.vocab)
TGT.vocab = TranslationVocab(bpe_tokenizer.vocab)

pad_idx = TGT.vocab.stoi[BLANK_WORD]

In [ ]:
# === DDP CHANGE ===
class TorchtextDatasetWrapper(Dataset):
    def __init__(self, dataset):
        self.examples = dataset.examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [ ]:
# === DDP CHANGE ===
def collate_fn(batch, pad_idx):
    # === 1. Sort by source length (descending) ===
    #batch = sorted(batch, key=lambda x: len(x.src), reverse=True)

    src_batch = []
    trg_batch = []

    for ex in batch:
        src_batch.append(ex.src)
        trg_batch.append(ex.trg)

    # === 2. Pad tokens & Numericalize ===
    src_padded = SRC.pad(src_batch)
    trg_padded = TGT.pad(trg_batch)

    src_tokens = SRC.numericalize(src_padded).squeeze(-1).transpose(0,1)
    trg_tokens = TGT.numericalize(trg_padded).squeeze(-1).transpose(0,1)

    # === 3. HARD TOKEN CAP (prevents OOM) ===
    #MAX_TOKENS = 6250  # try 2000 if still OOM

    #B, L = trg_tokens.shape

    #if B * L > MAX_TOKENS:
     #   new_B = max(1, MAX_TOKENS // L)  # ensure at least 1 example
      #  src_tokens = src_tokens[:new_B]
       # trg_tokens = trg_tokens[:new_B]

    # === 4. Return standard Batch object ===
    return Batch(src_tokens, trg_tokens, pad_idx)

In [ ]:
def train_epoch(model, loader, loss_compute, device):
    model.train()

    total_loss = torch.tensor(0.0, device=device)
    total_tokens = torch.tensor(0.0, device=device)

    for batch in loader:
        batch.src = batch.src.to(device)
        batch.trg = batch.trg.to(device)
        batch.src_mask = batch.src_mask.to(device)
        batch.trg_mask = batch.trg_mask.to(device)
        batch.trg_y = batch.trg_y.to(device)

        out = model(batch.src, batch.trg, batch.src_mask, batch.trg_mask)

        loss = loss_compute(out, batch.trg_y, batch.ntokens)

        total_loss += loss
        total_tokens += batch.ntokens

    # === DDP CHANGE === global reduction
    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
    dist.all_reduce(total_tokens, op=dist.ReduceOp.SUM)

    return (total_loss / total_tokens).item()

In [ ]:
def validate(model, loader, loss_compute, device):
    model.eval()

    total_loss = torch.tensor(0.0, device=device)
    total_tokens = torch.tensor(0.0, device=device)

    with torch.no_grad():
        for batch in loader:
            batch.src = batch.src.to(device)
            batch.trg = batch.trg.to(device)
            batch.src_mask = batch.src_mask.to(device)
            batch.trg_mask = batch.trg_mask.to(device)
            batch.trg_y = batch.trg_y.to(device)

            out = model(batch.src, batch.trg, batch.src_mask, batch.trg_mask)

            loss = loss_compute(out, batch.trg_y, batch.ntokens)

            total_loss += loss
            total_tokens += batch.ntokens

    # === DDP CHANGE ===
    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
    dist.all_reduce(total_tokens, op=dist.ReduceOp.SUM)

    return (total_loss / total_tokens).item()

In [ ]:

# === DDP CHANGE ===
local_rank = setup_distributed()
device = torch.device(f"cuda:{local_rank}")

cwd = os.getcwd()
# tokenizer
bpe_tokenizer = BPETokenizer()
bpe_tokenizer.load(f"{cwd}/wmt_ruleset.json")

vocab_size = len(bpe_tokenizer.vocab)
pad_idx = bpe_tokenizer.vocab.index("<blank>")

# model
model = make_model(vocab_size, vocab_size, N=6).to(device)

# === DDP CHANGE === wrap model
model = DDP(model, device_ids=[local_rank])

# optimizer
optimizer = torch.optim.Adam(
    model.module.parameters(),  # === DDP CHANGE ===
    lr=0,
    betas=(0.9, 0.98),
    eps=1e-9,
)

# === DDP CHANGE === scale LR
model_opt = NoamOpt(
    model.module.src_embed[0].d_model,
    factor=1 * dist.get_world_size(),
    warmup=4000,
    optimizer=optimizer,
)

criterion = LabelSmoothing(vocab_size, pad_idx).to(device)

# === DDP CHANGE === dataset + sampler
# === DDP CHANGE === dataset + sampler

train_examples = load_parallel_corpus(f"{cwd}/wmt14/wmt14.train.en", f"{cwd}/wmt14/wmt14.train.de", shuffle=True)
val_examples   = load_parallel_corpus(f"{cwd}/wmt14/wmt14.validation.en", f"{cwd}/wmt14/wmt14.validation.de")

train = data.Dataset(train_examples, fields=[('src', SRC), ('trg', TGT)])
val   = data.Dataset(val_examples, fields=[('src', SRC), ('trg', TGT)])

train.examples = [ex for ex in train.examples if filter_pred(ex)]
val.examples   = [ex for ex in val.examples if filter_pred(ex)]

train_dataset = TorchtextDatasetWrapper(train)
val_dataset   = TorchtextDatasetWrapper(val)

train_sampler = DistributedSampler(train_dataset, shuffle=False)
val_sampler   = DistributedSampler(val_dataset, shuffle=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=113,
    sampler=train_sampler,
    collate_fn=lambda x: collate_fn(x, pad_idx)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=113,
    sampler=val_sampler,
    collate_fn=lambda x: collate_fn(x, pad_idx)
)

os.makedirs(f"{cwd}/checkpoints42", exist_ok=True)

for epoch in range(20):
    # === DDP CHANGE === shuffle per epoch
    train_sampler.set_epoch(epoch)

    train_loss = train_epoch(
        model,
        train_loader,
        SimpleLossCompute(model.module.generator, criterion, model_opt),
        device,
    )
    val_loss = validate(
        model,
        val_loader,
        SimpleLossCompute(model.module.generator, criterion, None),
        device,
    )

    # === DDP CHANGE === sync before checkpoint
    dist.barrier()

    if is_main():
        print(f"Epoch {epoch} Train Loss: {train_loss}")
        print(f"Epoch {epoch} Val Loss: {val_loss}")

        torch.save({
            "epoch": epoch,
            "model": model.module.state_dict(),
            "optimizer": model_opt.optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
        }, f"checkpoints42/epoch_{epoch}.pt")

cleanup_distributed()

# BLEU Evaluation

In [ ]:
from torch.autograd import Variable
from collections import Counter
import math
import re

In [ ]:

def translate_sentence(model, sentence, src_vocab, tgt_vocab, bpe_tokenizer, max_len=50, beam_size=5, alpha=0.6, device=None):
    

    tokens = bpe_tokenizer.tokenize(sentence)
    src_indices = [src_vocab.stoi[tok] for tok in tokens]
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)  # [1, seq_len]
    src_mask = torch.ones(1, 1, src_tensor.size(1)).type_as(src_tensor)
    
    start_symbol = tgt_vocab.stoi[BOS_WORD]
    eos_symbol = tgt_vocab.stoi[EOS_WORD]

    translated_indices = beam_search_decode(model, src_tensor, src_mask,
                                               max_len=max_len,
                                               start_symbol=start_symbol,
                                               beam_size=beam_size,
                                               alpha=alpha,
                                               eos_symbol=eos_symbol)
    # translated_indices = greedy_decode(model, src_tensor, src_mask,
    #                                   max_len=max_len, start_symbol=start_symbol, eos_symbol=eos_symbol)
    
    translated_tokens = [tgt_vocab.itos[idx.item()] for idx in translated_indices[0]]  # batch=0

    
    return translated_tokens

In [ ]:
# Example usage:
device = torch.device("cpu")
cwd = os.getcwd()

# Load tokenizer
bpe_tokenizer = BPETokenizer()
bpe_tokenizer.load(f"{cwd}/wmt_ruleset.json")
vocab = TranslationVocab(bpe_tokenizer.vocab)
vocab_size = len(vocab)
pad_idx = bpe_tokenizer.vocab.index(BLANK_WORD)

# Load model
model = make_model(vocab_size, vocab_size, N=6).to(device)
checkpoint = torch.load(f"{cwd}/checkpoints42/epoch_19.pt", map_location=device) # Adjust path as needed
model.load_state_dict(checkpoint['model'])
print(checkpoint['val_loss'])
model.eval()

#Translate a sample sentence
sentence = "Two sets of lights so close to one another: intentional or just a silly error?"

print("Original sentence:", sentence)
translation = translate_sentence(model, sentence, vocab, vocab, bpe_tokenizer)
print("Translation:", bpe_tokenizer.detokenize(translation))


In [ ]:
def bleu_score(references, hypotheses, max_n=4):

    weights = [0.25]*max_n

    clipped_counts = [0]*max_n
    total_counts = [0]*max_n

    ref_length = 0
    hyp_length = 0

    for ref, hyp in zip(references, hypotheses):

        ref_tokens = ref.split()
        hyp_tokens = hyp.split()

        ref_length += len(ref_tokens)
        hyp_length += len(hyp_tokens)

        for n in range(1, max_n+1):

            ref_ngrams = Counter(
                tuple(ref_tokens[i:i+n])
                for i in range(len(ref_tokens)-n+1)
            )

            hyp_ngrams = Counter(
                tuple(hyp_tokens[i:i+n])
                for i in range(len(hyp_tokens)-n+1)
            )

            total_counts[n-1] += sum(hyp_ngrams.values())

            for ng in hyp_ngrams:
                clipped_counts[n-1] += min(
                    hyp_ngrams[ng],
                    ref_ngrams.get(ng,0)
                )

    precisions = []

    for i in range(max_n):
        if total_counts[i] == 0:
            precisions.append(0)
        else:
            precisions.append(clipped_counts[i]/total_counts[i])

    if min(precisions) == 0:
        return 0

    score = sum(w*math.log(p) for w,p in zip(weights,precisions))

    bp = 1 if hyp_length > ref_length else math.exp(1-ref_length/hyp_length)

    return bp * math.exp(score)

In [ ]:
# --- Moses 13a-style tokenizer (approximation) ---
def moses_tokenize_13a(text):
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)

    # Separate punctuation
    text = re.sub(r"([.,!?;:@#\$%&\(\)\[\]\{\}<>\"'])", r" \1 ", text)

    # Separate dashes
    text = re.sub(r"(-)", r" \1 ", text)

    # Normalize spaces again
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
def compute_bleu_wmt14(model,
                       src_file,
                       tgt_file,
                       src_vocab,
                       tgt_vocab,
                       bpe_tokenizer,
                       device):

    hypotheses = []
    references = []

    model.eval()

    with torch.no_grad():
        with open(src_file, encoding="utf-8") as src_f, \
             open(tgt_file, encoding="utf-8") as tgt_f:

            for src_line, tgt_line in zip(src_f, tgt_f):

                src_sentence = src_line.strip()
                ref_sentence = tgt_line.strip()

                pred_sentence = translate_sentence(
                    model,
                    src_sentence,
                    src_vocab,
                    tgt_vocab,
                    bpe_tokenizer,
                    device=device
                )
                pred_sentence = bpe_tokenizer.detokenize(pred_sentence)
                # --- Apply Moses tokenization to BOTH sides ---
                pred_tok = moses_tokenize_13a(pred_sentence)
                ref_tok  = moses_tokenize_13a(ref_sentence)

                hypotheses.append(pred_tok)
                references.append(ref_tok)

    bleu = bleu_score(references, hypotheses)

    return bleu

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

cwd = os.getcwd() 
bpe_tokenizer = BPETokenizer()
bpe_tokenizer.load(f"{cwd}/wmt_ruleset.json")
vocab = TranslationVocab(bpe_tokenizer.vocab)
vocab_size = len(vocab)
pad_idx = bpe_tokenizer.vocab.index(BLANK_WORD)
model = make_model(vocab_size, vocab_size, N=6).to(device)
checkpoint = torch.load(f"{cwd}/checkpoints42/epoch_19.pt", map_location=device)
model.load_state_dict(checkpoint['model'])
print(checkpoint['val_loss'])
model.eval()

bleu = compute_bleu_wmt14(
    model,
    f"{cwd}/wmt14/wmt14.test.en",
    f"{cwd}/wmt14/wmt14.test.de",
    vocab,
    vocab,
    bpe_tokenizer,
    device
)

print(f"BLEU score (WMT14 de->en): {bleu:.4f}")